# Exploratory Data Analysis: Waymo Dataset
This notebook demonstrates custom logic for interacting with the tfrecords, visualizing bounding boxes, and understanding object class distributions.

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import glob
import os

# We'll assume scripts/utils.py is available or we add it to the path
import sys
sys.path.append('../scripts')
from utils import get_dataset

## 1. Helper function for visualizing instances

In [ ]:
def display_instances(batch, max_images=5):
    """
    Custom implementation to extract images and bounding boxes from a parsed record batch,
    and plot them using matplotlib.
    """
    images = batch['image'].numpy()
    boxes = batch['groundtruth_boxes'].numpy()
    classes = batch['groundtruth_classes'].numpy()
    
    num_images = min(len(images), max_images)
    fig, axes = plt.subplots(num_images, 1, figsize=(15, 8 * num_images))
    if num_images == 1:
        axes = [axes]
        
    colors = {1: 'red', 2: 'blue', 4: 'green'} # 1: vehicle, 2: pedestrian, 4: cyclist
    
    for idx in range(num_images):
        ax = axes[idx]
        ax.imshow(images[idx])
        h, w, _ = images[idx].shape
        
        # Draw boxes
        for box, cls in zip(boxes[idx], classes[idx]):
            if cls == 0: continue # Skip padded elements
            ymin, xmin, ymax, xmax = box
            color = colors.get(cls, 'yellow')
            rect = patches.Rectangle((xmin * w, ymin * h), (xmax - xmin) * w, (ymax - ymin) * h,
                                     linewidth=2, edgecolor=color, facecolor='none')
            ax.add_patch(rect)
        ax.axis('off')
        ax.set_title(f'Sample {idx+1}')
    plt.tight_layout()
    plt.show()

## 2. Load dataset and visualize

In [ ]:
# Update the path below to map to your data folder
tfrecord_files = glob.glob('../data/train/*.tfrecord')
if tfrecord_files:
    dataset = get_dataset(tfrecord_files[0]) # Get the first dataset
    batch = next(iter(dataset)) # iterate
    display_instances(batch, max_images=3)
else:
    print('No TFRecords found. Please generate them first.')

## 3. Data Distribution Analysis
We need to understand class imbalance. Let's count the occurrences of each class over a few tfrecords.

In [ ]:
from collections import Counter
class_counts = Counter()

if tfrecord_files:
    print("Analyzing data distribution...")
    # Limit to 5 files to avoid excessive time during EDA
    for f in tfrecord_files[:5]:
        ds = get_dataset(f)
        for elem in ds:
            classes = elem['groundtruth_classes'].numpy().flatten()
            class_counts.update(classes[classes > 0]) # mask out zeros
            
    labels = ["Vehicle (1)", "Pedestrian (2)", "Cyclist (4)"]
    counts = [class_counts.get(1, 0), class_counts.get(2, 0), class_counts.get(4, 0)]
    
    plt.figure(figsize=(8,5))
    plt.bar(labels, counts, color=['red', 'blue', 'green'])
    plt.title('Object Class Distribution in Sample')
    plt.ylabel('Count')
    plt.show()
    print(f"Raw Class Counts: {class_counts}")